In [ ]:
#| default_exp llm

# llm

> LLM client wrapping litellm. Handles chunking, JSON extraction, and retries.
>
> Model is configured via `LLMConfig`. API keys come from environment variables — never from code.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import json
import re
import time
from typing import Any

import litellm
from rich.console import Console
from manhualizer.config import LLMConfig
from manhualizer.prompts import TemplateSet

_console = Console()

## Text Chunking

In [ ]:
#| export
def chunk_story(
    text: str,
    max_tokens: int = 3000,
    tokens_per_word: float = 1.4,
) -> list[str]:
    """Split story text into chunks that fit within `max_tokens`.

    Strategy (in order of preference):
    1. Split on double newlines (paragraph breaks)
    2. Split on single newlines
    3. Split on sentence boundaries ('. ', '! ', '? ')
    4. Hard word-count split as last resort

    Chunks are joined greedily until the token budget is exhausted.
    """
    max_words = int(max_tokens / tokens_per_word)

    # Try splits from coarsest to finest
    for separator in ["\n\n", "\n", r"(?<=[.!?]) +"]:
        parts = re.split(separator, text.strip())
        if len(parts) > 1:
            return _join_parts(parts, max_words)

    # Fallback: hard word-count split
    words = text.split()
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]


def _join_parts(parts: list[str], max_words: int) -> list[str]:
    """Greedily join parts into chunks that don't exceed max_words."""
    chunks, current, current_words = [], [], 0
    for part in parts:
        part = part.strip()
        if not part:
            continue
        part_words = len(part.split())
        if current and current_words + part_words > max_words:
            chunks.append(" ".join(current))
            current, current_words = [], 0
        current.append(part)
        current_words += part_words
    if current:
        chunks.append(" ".join(current))
    return chunks

## JSON Extraction

In [ ]:
#| export
def extract_json(text: str) -> Any:
    """Extract and parse JSON from an LLM response.

    Handles responses that are:
    - Pure JSON
    - JSON wrapped in markdown fences (```json ... ```)
    - JSON embedded in surrounding prose

    Raises ValueError if no valid JSON can be found.
    """
    # Try direct parse first
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strip markdown fences
    fenced = re.search(r"```(?:json)?\s*([\s\S]+?)\s*```", text)
    if fenced:
        try:
            return json.loads(fenced.group(1))
        except json.JSONDecodeError:
            pass

    # Find the largest {...} or [...] block
    for pattern in (r"(\{[\s\S]+\})", r"(\[[\s\S]+\])"):
        match = re.search(pattern, text)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                pass

    raise ValueError(f"Could not extract JSON from LLM response:\n{text[:500]}")

## LLM Client

In [ ]:
#| export
class LLMClient:
    """Thin wrapper around litellm for manhualizer pipeline calls.

    All calls go through `complete()`. JSON responses are parsed via `complete_json()`.
    API keys are read from environment variables by litellm automatically.
    """

    def __init__(self, config: LLMConfig, templates: TemplateSet):
        self.config = config
        self.templates = templates
        litellm.drop_params = True  # ignore unsupported params per model

    def complete(
        self,
        user_prompt: str,
        system_prompt: str = "",
        temperature: float | None = None,
        max_tokens: int | None = None,
    ) -> str:
        """Call the LLM and return the response text.

        Automatically retries on rate limit errors with exponential backoff
        (60 s, 120 s, 180 s) before giving up.
        """
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})

        kwargs = {
            "model": self.config.model,
            "messages": messages,
            "temperature": temperature if temperature is not None else self.config.temperature,
            "max_tokens": max_tokens if max_tokens is not None else self.config.max_tokens,
        }

        max_rate_retries = 3
        for attempt in range(max_rate_retries + 1):
            try:
                response = litellm.completion(**kwargs)
                return response.choices[0].message.content
            except (litellm.exceptions.RateLimitError, litellm.exceptions.InternalServerError) as e:
                if attempt >= max_rate_retries:
                    raise
                # 429 rate limit → long backoff; 529 overloaded → shorter backoff
                is_overloaded = isinstance(e, litellm.exceptions.InternalServerError)
                wait = 30 * (attempt + 1) if is_overloaded else 60 * (attempt + 1)
                label = "overloaded (529)" if is_overloaded else "rate limit (429)"
                _console.print(
                    f"[yellow]llm: {label} — waiting {wait}s before retry "
                    f"({attempt + 1}/{max_rate_retries})[/yellow]"
                )
                time.sleep(wait)

    def complete_json(
        self,
        user_prompt: str,
        system_prompt: str = "",
        retries: int = 2,
    ) -> Any:
        """Call the LLM and parse the response as JSON.

        Retries up to `retries` times on parse failure, asking the model to
        return only valid JSON.
        """
        last_error: Exception | None = None
        prompt = user_prompt
        for attempt in range(retries + 1):
            text = self.complete(prompt, system_prompt)
            try:
                return extract_json(text)
            except ValueError as e:
                last_error = e
                prompt = (
                    f"{user_prompt}\n\n"
                    f"Your previous response could not be parsed as JSON. "
                    f"Return ONLY valid JSON, no explanation, no markdown.\n"
                    f"Previous response (for reference):\n{text[:300]}"
                )
        raise ValueError(f"LLM failed to return valid JSON after {retries + 1} attempts: {last_error}")

    def complete_from_template(
        self,
        template_file: str,
        prompt_key: str,
        system_key: str = "system",
        as_json: bool = False,
        max_tokens: int | None = None,
        **variables: Any,
    ) -> str | Any:
        """Render a template and call the LLM.

        Args:
            template_file: YAML file name (e.g. 'analyze.yml')
            prompt_key: Key in the YAML for the user prompt template
            system_key: Key in the YAML for the system prompt (default 'system')
            as_json: If True, parse and return the response as JSON
            max_tokens: Override the default max_tokens for this call only
            **variables: Template substitution variables
        """
        user_prompt = self.templates.render(template_file, prompt_key, **variables)
        try:
            system_prompt = self.templates.get(template_file, system_key)
        except KeyError:
            system_prompt = ""

        if as_json:
            return self.complete_json(user_prompt, system_prompt, max_tokens=max_tokens)
        return self.complete(user_prompt, system_prompt, max_tokens=max_tokens)

## Tests

In [ ]:
# Chunking tests (no API calls needed)
text = "Paragraph one is here.\n\nParagraph two follows.\n\nParagraph three ends it."
chunks = chunk_story(text, max_tokens=20)
assert len(chunks) >= 1

# Single long paragraph falls back to sentence splitting
long = "First sentence. Second sentence. Third sentence. Fourth sentence."
chunks2 = chunk_story(long, max_tokens=10)
assert len(chunks2) >= 1

print(f"Chunking OK — {len(chunks)} chunks from paragraph text")

In [ ]:
# JSON extraction tests (no API calls needed)
assert extract_json('{"a": 1}') == {"a": 1}
assert extract_json('```json\n{"a": 1}\n```') == {"a": 1}
assert extract_json('Here is the result: {"a": 1} done.') == {"a": 1}
print("JSON extraction OK")

In [ ]:
# LLMClient construction test (no API call)
from manhualizer.config import LLMConfig
from manhualizer.prompts import load_templates

cfg = LLMConfig()
templates = load_templates("default")
client = LLMClient(cfg, templates)
assert client.config.model == "anthropic/claude-sonnet-4-6"
print("LLMClient construction OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()